In [2]:
!pip install langchain-community
!pip install pypdf
!pip install faiss-cpu
!pip install tiktoken
!pip install langchain_groq

In [3]:
import openai
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS

In [4]:
pdf = PyPDFLoader("/content/Power+BI+Ebook.pdf")
doc = pdf.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(doc)

In [5]:
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
db = FAISS.from_documents(documents=texts, embedding=embeddings)

/tmp/ipython-input-5-3579917082.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models

In [6]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model='llama3-70b-8192',
    temperature=0,
    max_tokens=None,
    api_key='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'
)

In [7]:
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

QUESTION_PROMPT = PromptTemplate.from_template("""Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question.

You can assume the question about Power BI

Chat History:
{chat_history}
Follow up Input: {question}
Standalon question:""")

qa = ConversationalRetrievalChain.from_llm(llm=llm,
                                           retriever=db.as_retriever(),
                                           condense_question_prompt=QUESTION_PROMPT,
                                           return_source_documents=True,
                                           verbose=False
                                           )

In [8]:
chat_history=[]
query = """?What are the components of Power BI?"""
result = qa({"question": query, "chat_history": chat_history})
print(result['answer'])

/tmp/ipython-input-8-3020259085.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa({"question": query, "chat_history": chat_history})


Based on the provided context, it appears that Power BI is a suite of tools, and one of its components is Power BI Desktop, which is a Windows application that provides features for data modeling, data preparation, data analysis, and data visualization.

However, the context does not provide a comprehensive list of all the components of Power BI. It only mentions Power BI Desktop and Power Query Editor, which is likely a part of Power BI Desktop.

Therefore, I don't know the complete list of components of Power BI based on the provided context.


In [9]:
sample_queries = [
    {
        "question": "What are the main components of Power BI?",
        "answer": "The main components of Power BI are Power BI Desktop, Power BI Service, and Power BI Mobile.",
        "reference": "Power BI has three main components: Desktop (Windows app for creating reports), Service (cloud platform for sharing and collaboration), and Mobile (app for viewing reports on mobile devices)."
    },
    {
        "question": "What are the key features of Power BI Desktop?",
        "answer": "Key features include data connection and transformation, data modeling, interactive visualizations, report publishing, and collaboration tools.",
        "reference": "Power BI Desktop allows users to get data, analyze it with DAX, visualize using 150+ visuals, publish to cloud or on-premises, and collaborate with team members."
    },
    {
        "question": "How does Power BI integrate with Excel?",
        "answer": "Power BI can analyze data in Excel, import Excel data, connect to Excel workbooks, and pin Excel ranges to dashboards.",
        "reference": "Power BI integrates with Excel by enabling analysis in Excel, data import, connection to workbooks, and uploading Excel files for dashboard pinning."
    },
    {
        "question": "What is the Power Query Editor used for?",
        "answer": "Power Query Editor is used for data transformation and cleansing before importing data into models or visualizations.",
        "reference": "Power Query Editor includes tools like ribbon tabs, data preview, query settings, and footer with data stats for transforming and shaping data."
    },
    {
        "question": "How can you group data in Power BI?",
        "answer": "You can group data using the 'Group By' feature in the Transform tab of the Query Editor by selecting columns and specifying aggregate calculations.",
        "reference": "In Power BI's Query Editor, the Group By dialog allows users to select columns and apply aggregate functions to group data."
    }
]


In [10]:
import pandas as pd
df = pd.DataFrame(sample_queries)
df.head(5)

,question,answer,reference
0,What are the main components of Power BI?,The main components of Power BI are Power BI D...,Power BI has three main components: Desktop (W...
1,What are the key features of Power BI Desktop?,Key features include data connection and trans...,"Power BI Desktop allows users to get data, ana..."
2,How does Power BI integrate with Excel?,"Power BI can analyze data in Excel, import Exc...",Power BI integrates with Excel by enabling ana...
3,What is the Power Query Editor used for?,Power Query Editor is used for data transforma...,Power Query Editor includes tools like ribbon ...
4,How can you group data in Power BI?,You can group data using the 'Group By' featur...,"In Power BI's Query Editor, the Group By dialo..."


In [11]:
retriever = db.as_retriever()

In [12]:
def process_query(query):
  chat_history = []
  result = qa({"question": query, "chat_history": chat_history})
  relevant_docs = retriever.invoke(query)
  print(result['answer'])
  return result['answer'], relevant_docs

In [13]:
process_query("What is the Power Query Editor used for?")[0][0]

The Power Query Editor is a data transformation and cleansing tool that is built into Power BI, Excel, and other Microsoft products. It provides users with a powerful set of tools for transforming and shaping their data before it is imported into a data model or visualization.


'T'

In [14]:
!pip install ragas

In [15]:
results = []

for _, row in df.iterrows():
  question = row['question']
  ground_truth = row['answer']

  answer, relevant_docs = process_query(question)

  results.append({
      "user_input": question,
      "reference": ground_truth,
      "response": answer,
      "retrieved_contexts": [relevant_docs[0].page_content]
  })

Based on the provided context, the main components of Power BI are not explicitly mentioned. However, it can be inferred that Power BI consists of several features and tools, including:

1. Power BI Desktop: A Windows application for creating interactive data visualizations, reports, and dashboards.
2. Power Query Editor: A tool for data preparation and modeling.
3. Data Analysis Expression (DAX): A language for creating formulas and calculations.
4. Power BI Relationships and KPI: Tools for creating relationships between data tables and defining key performance indicators.
5. Administration and Security: Features for managing user roles, permissions, and security settings.

Please note that this is not an exhaustive list, and Power BI may have additional components or features not mentioned in the provided context.
According to the provided context, the key features of Power BI Desktop are:

* Get Data: Easily connect, clean, and mashup data from 80+ data sources, both on-premises and

In [16]:
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

In [17]:
evaluation_dataset = EvaluationDataset.from_list(results)

In [18]:
evaluator_llm = LangchainLLMWrapper(llm)

In [19]:
ragas_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm = evaluator_llm
)

Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[5]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-70b-8192` in organization `org_01jzsg4hmre6cazkgbn1mm172n` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 10701, Requested 1207. Please try again in 59.088s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[13]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[10]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[14]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[11]: TimeoutError()


In [20]:
ragas_result

{'context_recall': 0.7100, 'faithfulness': 0.2857, 'factual_correctness(mode=f1)': nan}

In [21]:
!pip install rouge

In [22]:
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge

In [23]:
rouge = Rouge()
bleu = []
rouge_one = []

In [24]:
for item in results:
  reference = [item["reference"].split()]
  hypothesis = item["response"].split()
  bleu.append(sentence_bleu(reference, hypothesis))

  scores = rouge.get_scores(item["response"], item["reference"])[0]
  rouge_one.append(scores["rouge-1"]["f"])

/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

In [25]:
bleu

[0.05327351966022204,
 8.844844403089351e-232,
 1.4802510786745944e-78,
 0.10987931098593881,
 1.0788073578036947e-78]

In [26]:
rouge_one

[0.14432989489637582,
 0.07594936417240838,
 0.328358205168189,
 0.45283018446422224,
 0.2765957410955184]

In [27]:
!pip install bert_score

In [29]:
from transformers import BartForConditionalGeneration, BartTokenizer
from bert_score import BERTScorer
import torch

In [32]:
bart_model = BartForConditionalGeneration.from_pretrained('sshleifer/distilbart-cnn-12-6')
bart_tokenizer = BartTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')

bert_scorer = BERTScorer(lang='en',rescale_with_baseline=True)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [33]:
bert_scores = []
bart_scores = []

for item in results:
  inputs = bart_tokenizer(item["response"], return_tensors='pt', truncation=True, padding=True )
  with torch.no_grad():
    bart_score = bart_model(**inputs).logits
  bart_scores.append(bart_score.mean().item())

  P, R, F1 = bert_scorer.score([item["response"]], [item["reference"]])
  bert_scores.append(F1.numpy().mean())

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [34]:
print(bart_scores)
print(bert_scores)


[-0.6549558639526367, -0.820429265499115, -0.6552203297615051, -0.8795600533485413, -0.643166720867157]
[np.float32(0.1887799), np.float32(0.2375883), np.float32(0.2949637), np.float32(0.5012742), np.float32(0.3329664)]
